In [10]:
__author__ = 'Cyrille Mvomo, https://github.com/cyrillemvomo'
__version__ = "1"
__license__ = "MIT"

**Import**

In [11]:
import sys
sys.path.append("/Users/cyrilleetude/Documents/GitHub/WhiteBoxDL/Preprocessing/1_Get_Data/McGill_Cohort/1_Extract")
from Extracted_Trina import Extracted_Data_Trina
import numpy as np
import pandas as pd
import copy #so that I can copy object
import pickle

**Read**

In [12]:
EXTRACTED_DATA = Extracted_Data_Trina(Gait_Data_Path = "/Volumes/Active/Datasets/A_Principal_Component_Model_Provides_Gait_Features_for_Tracking_Symptoms_Severity_and_Neural_Progres/McGill_Cohort/Wearables/Steering", 
                                Medical_Record_Path = "/Volumes/Active/Datasets/WhiteBoxDL/Demographics/McGill_Trina/Demographic", 
                                Source_Folder_Path = "/Users/cyrilleetude/Documents/GitHub/WhiteBoxDL/Preprocessing/1_Get_Data/McGill_Cohort/0_Source").RawData_Extracted(save_data=False)


**Crop into 5 second bouts**

In [13]:
# Declare variables
sampling_rate= 128 #hz
duration_start_walking = 120 + 10 #number of seconds between injection of FDG (start of APDM data collection) and start of gait task + 10 second to remove start effect
gait_start = duration_start_walking * sampling_rate# convert in frames
gait_end = gait_start + (120*sampling_rate) # stop after 2 min (in frames)

In [14]:
# PRE CROP to only keep 2 min of interest (start at 130s and end at 250 s)
IDs = np.array(list(EXTRACTED_DATA.keys()))[1:]
for id in IDs:
    EXTRACTED_DATA[id] = EXTRACTED_DATA[id].loc[(EXTRACTED_DATA[id].index >= gait_start) 
                                                                  & (EXTRACTED_DATA[id].index <= gait_end)]


In [31]:
# Crop
COMMENTS = pd.read_excel("/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort/2_Transform/0_Signal_Processing/temp/Comments_Steering.xlsx")
EXTRACTED_DATA_CROPPED = copy.deepcopy(EXTRACTED_DATA) # to avoid modifying EXTRACTED_DATA_STEERING too
for id in IDs:
    if list(IDs).index(id) in list(COMMENTS.loc[COMMENTS["comments"] != "clean"].index):# if there is something to crop (i.e. not clean)
        EXTRACTED_DATA_CROPPED[id] = EXTRACTED_DATA_CROPPED[id].loc[EXTRACTED_DATA_CROPPED[id].index >= (210*sampling_rate)].iloc[:, :2].reset_index(drop=True) #reset the indexes and drop time column as useless for signal processing + only keep after 210s
        EXTRACTED_DATA_CROPPED[id]["TIME_S"] = EXTRACTED_DATA_CROPPED[id]["TIME_S"].values - EXTRACTED_DATA_CROPPED[id]["TIME_S"].values[0]
    else:
        EXTRACTED_DATA_CROPPED[id] = EXTRACTED_DATA_CROPPED[id].iloc[:, :2].reset_index(drop=True) #reset the indexes and time column
        EXTRACTED_DATA_CROPPED[id]["TIME_S"] = EXTRACTED_DATA_CROPPED[id]["TIME_S"].values - EXTRACTED_DATA_CROPPED[id]["TIME_S"].values[0]


In [41]:
# Split in 5 second bouts
EXTRACTED_DATA_SPLITTED = {}
for participant_id in list(EXTRACTED_DATA_CROPPED.keys())[1:]:
    five_s_bouts_indexes = np.array_split(EXTRACTED_DATA_CROPPED[participant_id].index.values, len(EXTRACTED_DATA_CROPPED[participant_id])//(5*sampling_rate))
    participant_data = []
    for bout_id in range(len(five_s_bouts_indexes)):
        participant_data.append(EXTRACTED_DATA_CROPPED[participant_id].iloc[five_s_bouts_indexes[bout_id]].ACC_VERTICAL_MS2.values)
    
    EXTRACTED_DATA_SPLITTED[participant_id] = participant_data

In [48]:
# Save as bin file 
with open("/Users/cyrilleetude/Documents/GitHub/WhiteBoxDL/Preprocessing/1_Get_Data/McGill_Cohort/2_Transform/temp/Extracted_Data_Trina.bin", "wb") as f:
            pickle.dump(EXTRACTED_DATA_SPLITTED, f)